# Notebook 04 — Faster R-CNN + MobileNetV3

**Dataset:** A (Drone/UAV) | **Paradigm:** Two-Stage RPN  
**Backbone:** MobileNetV3-Large (edge-optimized)

## Config
- Image: 416×416 | Batch: 4 | Epochs: 50 | Optimizer: SGD lr=0.005 | FP16: ❌

> **Colab Install Note:** Detectron2 requires a specific install command — see cell below.

In [ ]:
# ── Install Detectron2 (Colab — run once) ─────────────────────────────────
# import torch
# TORCH = torch.__version__.split('+')[0].replace('.', '')
# CUDA  = torch.version.cuda.replace('.', '')
# !pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu{CUDA}/torch{TORCH}/index.html --quiet

In [ ]:
import torch
import detectron2
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.data.datasets import register_coco_instances
from pathlib import Path

ROOT        = Path('..')
DATA_A      = ROOT / 'data' / 'dataset-a'
RESULTS_DIR = ROOT / 'results' / 'experiment-a' / 'faster_rcnn'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"Detectron2: {detectron2.__version__}")

In [ ]:
# ── Register Datasets ──────────────────────────────────────────────────────
register_coco_instances(
    "fire_train", {},
    str(DATA_A / "annotations" / "train.json"),
    str(DATA_A / "images" / "train" / "images")
)
register_coco_instances(
    "fire_val", {},
    str(DATA_A / "annotations" / "val.json"),
    str(DATA_A / "images" / "val" / "images")
)
print("Datasets registered.")

In [ ]:
# ── Config: Faster R-CNN + MobileNetV3 ────────────────────────────────────
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(
    "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"
))
# Swap backbone to MobileNetV3
# Note: requires torchvision MobileNetV3 backbone registration
cfg.MODEL.BACKBONE.NAME = "build_mnv3_large_fpn_backbone"  # custom — see src/models/faster_rcnn.py
cfg.DATASETS.TRAIN          = ("fire_train",)
cfg.DATASETS.TEST           = ("fire_val",)
cfg.DATALOADER.NUM_WORKERS  = 2
cfg.MODEL.WEIGHTS           = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml")
cfg.SOLVER.IMS_PER_BATCH    = 4
cfg.SOLVER.BASE_LR          = 0.005
cfg.SOLVER.MAX_ITER         = 5000   # ~50 epochs for typical dataset
cfg.SOLVER.STEPS            = (3500, 4500)
cfg.SOLVER.GAMMA            = 0.1
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 2  # fire, smoke
cfg.INPUT.MIN_SIZE_TRAIN    = (416,)
cfg.INPUT.MAX_SIZE_TRAIN    = 416
cfg.OUTPUT_DIR              = str(RESULTS_DIR)

print(cfg)

In [ ]:
# ── Train ──────────────────────────────────────────────────────────────────
trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

In [ ]:
# ── Evaluate ───────────────────────────────────────────────────────────────
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

evaluator = COCOEvaluator("fire_val", output_dir=str(RESULTS_DIR))
val_loader = build_detection_test_loader(cfg, "fire_val")
results = inference_on_dataset(trainer.model, val_loader, evaluator)
print(results)

In [ ]:
# ── FLOPs Measurement ──────────────────────────────────────────────────────
from thop import profile
import torch

# Load as torchvision model for FLOPs measurement
import torchvision
model_tv = torchvision.models.detection.fasterrcnn_mobilenet_v3_large_320_fpn(pretrained=False, num_classes=3)
dummy_input = torch.zeros(1, 3, 416, 416)
macs, params = profile(model_tv, inputs=([dummy_input],), verbose=False)
print(f"Parameters: {params/1e6:.2f}M")
print(f"GFLOPs:     {macs*2/1e9:.2f}")